# 05 Graph

## 1. Import the needed libraries

In [54]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Face import Face
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Helper import Helper

## 2. Check the TopologicPy version

In [55]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This tutorial requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.18) is OLDER than the latest version (0.9.21) from PyPI. Please consider upgrading to the latest version.


## 3. Set your renderer
* Visual Studio Code: `"vscode"`
* Google Colab: `"colab"`
* Browser: `"browser"`

In [56]:
renderer = "vscode"

In [57]:
import os
os.makedirs("Images", exist_ok=True)

## 4. Importing the OBJs

In [58]:
OBJ_PATH = r"C:\Users\etmaglari\IAAC\etmaglari_gML\R01\Rhino_Geometry _etm.obj"

objects  = Topology.ByOBJPath(OBJ_PATH, selfMerge=False)
topology = objects[0] if isinstance(objects, list) else objects
print(f"Loaded: {type(topology).__name__}")

clean = Topology.RemoveCoplanarFaces(
    topology,
    epsilon=0.1,
    tolerance=0.001
)
print(f"Faces before: {len(Topology.Faces(topology))}")
print(f"Faces after RemoveCoplanarFaces: {len(Topology.Faces(clean))}")

Loaded: Cluster
Faces before: 344
Faces after RemoveCoplanarFaces: 22


## 5. Show the geometry

In [59]:
fig = Topology.Show(
    clean,
    faceColor=[210, 210, 250],
    faceOpacity=0.6,
    edgeColor="white",
    edgeWidth=2,
    showVertices=False,
    backgroundColor="black",
    width=800,
    height=600,
    renderer=renderer
)
if fig:
    fig.write_html("Images/01_geometry.html")

## 6. Build CellComplex from cleaned geometry

In [60]:
faces = Topology.Faces(clean)
cc = CellComplex.ByFaces(faces, tolerance=0.001)
cells = Topology.Cells(cc)
print(f"CellComplex: {len(cells)} cells, {len(Topology.Faces(cc))} faces, {len(Topology.Edges(cc))} edges")

CellComplex: 10 cells, 58 faces, 105 edges


## 7. Primal Graph

Vertices at the geometric corners of the CellComplex; edges along wall boundaries.

In [61]:
primal_verts = Topology.Vertices(cc)
primal_edges = Topology.Edges(cc)
print(f"Primal — Vertices: {len(primal_verts)}, Edges: {len(primal_edges)}")

fig = Topology.Show(
    cc,
    faceColor=[210, 210, 250], faceOpacity=0.15,
    edgeColor="red", edgeWidth=2,
    vertexColor="red", vertexSize=6,
    showVertices=True,
    backgroundColor="black",
    width=800, height=600,
    renderer=renderer
)
if fig:
    fig.write_html("Images/02_primal_graph.html")

Primal — Vertices: 58, Edges: 105


## 8. Dual Graph

One vertex per room (cell centroid); an edge between every pair of rooms that share a wall face.

In [62]:
g_dual = Graph.ByTopology(cc)
print(f"Dual — Nodes: {len(Graph.Vertices(g_dual))}, Edges: {len(Graph.Edges(g_dual))}")

for v in Graph.Vertices(g_dual):
    Topology.SetDictionary(v, Dictionary.ByKeysValues(["size", "color"], [18, "red"]))
for e in Graph.Edges(g_dual):
    Topology.SetDictionary(e, Dictionary.ByKeysValues(["width", "color"], [3, "white"]))

fig = Topology.Show(
    cc, g_dual,
    faceColor="white", faceOpacity=0.05,
    edgeColor="white", edgeWidth=0.5,
    vertexSizeKey="size", vertexColorKey="color",
    edgeWidthKey="width", edgeColorKey="color",
    showVertices=True,
    backgroundColor="black",
    width=800, height=600,
    renderer=renderer
)
if fig:
    fig.write_html("Images/03_dual_graph.html")

Dual — Nodes: 10, Edges: 16


## 9. Load Windows

In [63]:
WIN_PATH = r"C:\\Users\\etmaglari\\IAAC\\etmaglari_gML\\R01\\Rhino_Windows_etm.obj"

win_objects = Topology.ByOBJPath(WIN_PATH, selfMerge=False)
win_cluster = win_objects[0] if isinstance(win_objects, list) else win_objects
windows     = Topology.Faces(win_cluster)

for w in windows:
    Topology.SetDictionary(w, Dictionary.ByKeysValues(["color"], ["#4CC9F0"]))

print(f"Windows found: {len(windows)}")

fig = Topology.Show(
    clean, win_cluster,
    faceColor="#4CC9F0", faceOpacity=0.25,
    edgeColor="white", edgeWidth=0.5,
    showVertices=False,
    backgroundColor="black",
    width=800, height=600,
    renderer=renderer
)
if fig:
    fig.write_html("Images/04_windows.html")

Windows found: 13


## 10. Load Doors

In [64]:
DOOR_PATH = r"C:\\Users\\etmaglari\\IAAC\\etmaglari_gML\\R01\\Rhino_Doors_ETM.obj"

door_objects = Topology.ByOBJPath(DOOR_PATH, selfMerge=False)
door_cluster = door_objects[0] if isinstance(door_objects, list) else door_objects
doors        = Topology.Faces(door_cluster)

for d in doors:
    Topology.SetDictionary(d, Dictionary.ByKeysValues(["color"], ["#4CC9F0"]))

print(f"Doors found: {len(doors)}")

fig = Topology.Show(
    clean, door_cluster,
    faceColor="#4CC9F0", faceOpacity=0.25,
    edgeColor="white", edgeWidth=0.5,
    showVertices=False,
    backgroundColor="black",
    width=800, height=600,
    renderer=renderer
)
if fig:
    fig.write_html("Images/05_doors.html")

Doors found: 10


## 11. Primal Graph with Apertures

Overlay the primal graph on the room model and include both door and window geometry for reference.

In [65]:
aperture_cluster = Cluster.ByTopologies(windows + doors)
primal_verts = Topology.Vertices(cc)
primal_edges = Topology.Edges(cc)
primal_cluster = Cluster.ByTopologies(primal_edges + primal_verts)

print(f"Primal — Vertices: {len(primal_verts)}, Edges: {len(primal_edges)}")

fig = Topology.Show(
    cc, aperture_cluster, primal_cluster,
    faceColor="#4CC9F0", faceOpacity=0.12,
    edgeColor="red", edgeWidth=2,
    vertexColor="red", vertexSize=6,
    showVertices=True,
    backgroundColor="black",
    width=900, height=700,
    renderer=renderer
)
if fig:
    fig.write_html("Images/06_primal_with_apertures.html")

Primal — Vertices: 58, Edges: 105


## 12. Add Apertures to CellComplex

In [66]:
all_apertures = windows + doors
all_aperture_cluster = Cluster.ByTopologies(all_apertures)
for apt in all_apertures:
    Topology.SetDictionary(apt, Dictionary.ByKeysValues(["color"], ["#4CC9F0"]))

cc_apts = Topology.AddApertures(cc, all_apertures)

all_cells = Topology.Cells(cc_apts)
palette   = ["#E63946","#F4A261","#2A9D8F","#457B9D","#A8DADC",
             "#264653","#8338EC","#3A86FF","#FB5607","#FFBE0B"]
for i, cell in enumerate(all_cells):
    color = palette[i % len(palette)]
    Topology.SetDictionary(cell, Dictionary.ByKeysValues(["color", "id"], [color, i]))

print(f"CellComplex: {len(all_cells)} rooms, {len(all_apertures)} apertures")

fig = Topology.Show(
    cc_apts, all_aperture_cluster,
    faceColor="#4CC9F0", faceOpacity=0.12,
    edgeColor="white", edgeWidth=0.5,
    showVertices=False,
    backgroundColor="black",
    width=800, height=600,
    renderer=renderer
)
if fig:
    fig.write_html("Images/07_cellcomplex_apertures.html")

CellComplex: 10 rooms, 23 apertures


## 13. Access + Adjacency Graph

Rooms connected when they **share a wall** (direct=True), share an **interior aperture** (viaSharedApertures=True), or have an **exterior aperture** (toExteriorApertures=True). Node size scales with room surface area.

In [67]:
m = Graph.ByTopology(
    cc_apts,
    direct=True,
    viaSharedApertures=True,
    toExteriorApertures=True
)
print(f"Graph vertices: {len(Graph.Vertices(m))}")
print(f"Edges:          {len(Graph.Edges(m))}")

areas    = [Cell.SurfaceArea(cell) for cell in all_cells]
min_area = min(areas)
max_area = max(areas)

for v in Graph.Vertices(m):
    vx, vy, vz = Vertex.X(v), Vertex.Y(v), Vertex.Z(v)
    best_color = "gray"
    best_size  = 8
    best_dist  = float("inf")
    for cell, area in zip(all_cells, areas):
        c    = Topology.Centroid(cell)
        dist = ((vx - Vertex.X(c))**2 + (vy - Vertex.Y(c))**2 + (vz - Vertex.Z(c))**2)**0.5
        if dist < best_dist:
            best_dist  = dist
            cd         = Topology.Dictionary(cell)
            best_color = Dictionary.ValueAtKey(cd, "color") if cd else "gray"
            if best_color == "gray":
                best_size = 8
            else:
                best_size = 12 + int(48 * (area - min_area) / (max_area - min_area + 1))
    Topology.SetDictionary(v, Dictionary.ByKeysValues(["size", "color"], [best_size, best_color]))

for e in Graph.Edges(m):
    Topology.SetDictionary(e, Dictionary.ByKeysValues(["width", "color"], [2, "black"]))

fig = Topology.Show(
    [cc_apts, m, win_cluster, door_cluster],
    vertexSizeKey="size",
    vertexColorKey="color",
    edgeWidthKey="width",
    edgeColorKey="color",
    faceOpacity=0.15,
    backgroundColor="white",
    width=1000, height=1000,
    renderer=renderer
)
if fig:
    fig.write_html("Images/08_access_adjacency_graph.html")

Graph vertices: 10
Edges:          16


## 14. Room Centroid to Window and Door Graph

In [68]:
from collections import defaultdict

cells_cc = Topology.Cells(cc_apts)

def dist3(a, b):
    return ((a[0]-b[0])**2 + (a[1]-b[1])**2 + (a[2]-b[2])**2)**0.5

def centroid_xyz(topology):
    c = Topology.Centroid(topology)
    return (c.X(), c.Y(), c.Z())

# Room centroid nodes: same color/size logic as section 13 (area-scaled size)
areas = [Cell.SurfaceArea(cell) for cell in cells_cc]
min_area = min(areas)
max_area = max(areas)

room_verts = []
room_centroids = []
for cell, area in zip(cells_cc, areas):
    rc = Topology.Centroid(cell)
    room_centroids.append((rc.X(), rc.Y(), rc.Z()))
    cd = Topology.Dictionary(cell)
    color = Dictionary.ValueAtKey(cd, "color") if cd else "gray"
    if color == "gray":
        size = 8
    else:
        size = 12 + int(48 * (area - min_area) / (max_area - min_area + 1))
    v = Vertex.ByCoordinates(rc.X(), rc.Y(), rc.Z())
    Topology.SetDictionary(v, Dictionary.ByKeysValues(["size", "color"], [size, color]))
    room_verts.append(v)

# Aperture centroid nodes (small blue)
all_aperture_faces = windows + doors
all_aperture_types = ["window"] * len(windows) + ["door"] * len(doors)
aperture_verts = []
aperture_centroids = []
for apt in all_aperture_faces:
    ac = Topology.Centroid(apt)
    ap = (ac.X(), ac.Y(), ac.Z())
    aperture_centroids.append(ap)
    v = Vertex.ByCoordinates(*ap)
    Topology.SetDictionary(v, Dictionary.ByKeysValues(["size", "color"], [7, "#4CC9F0"]))
    aperture_verts.append(v)

# Build mapping: aperture index -> set of room indices (supports shared apertures)
apt_to_rooms = defaultdict(set)

# Primary strategy: use aperture-host faces in cc_apts and adjacent cells
for f in Topology.Faces(cc_apts):
    apts_on_face = Topology.ApertureTopologies(f, subTopologyType="face") or []
    if len(apts_on_face) == 0:
        continue
    adjacent_cells = Topology.SuperTopologies(f, hostTopology=cc_apts, topologyType="cell") or []
    room_indices = []
    for c in adjacent_cells:
        ccnt = centroid_xyz(c)
        ri = min(range(len(room_centroids)), key=lambda i: dist3(ccnt, room_centroids[i]))
        room_indices.append(ri)
    for apt in apts_on_face:
        ac = centroid_xyz(apt)
        aj = min(range(len(aperture_centroids)), key=lambda j: dist3(ac, aperture_centroids[j]))
        for ri in room_indices:
            apt_to_rooms[aj].add(ri)

# Fallback strategy: allow near-equidistant multi-room matches
for j, ap in enumerate(aperture_centroids):
    # Minimum face-centroid distance per room
    per_room = []
    for ri, cell in enumerate(cells_cc):
        face_centroids = [centroid_xyz(f) for f in (Topology.Faces(cell) or [])]
        if len(face_centroids) == 0:
            continue
        dmin = min(dist3(ap, fc) for fc in face_centroids)
        per_room.append((ri, dmin))

    if len(per_room) == 0:
        continue

    global_min = min(d for _, d in per_room)
    near_rooms = [ri for ri, d in per_room if d <= global_min + 0.25]

    # Always guarantee at least one room
    if len(apt_to_rooms[j]) == 0:
        for ri in near_rooms:
            apt_to_rooms[j].add(ri)
    # If an aperture is likely shared, keep multiple links
    elif len(apt_to_rooms[j]) == 1 and len(near_rooms) > 1:
        for ri in near_rooms:
            apt_to_rooms[j].add(ri)

# Build edges: connect each aperture centroid to all mapped room centroids
edges_ra = []
for aj, room_idxs in apt_to_rooms.items():
    av = aperture_verts[aj]
    for ri in sorted(room_idxs):
        rv = room_verts[ri]
        e = Edge.ByVertices([rv, av])
        Topology.SetDictionary(e, Dictionary.ByKeysValues(["width", "color"], [2, "#4CC9F0"]))
        edges_ra.append(e)

# Reporting
room_to_windows = defaultdict(int)
room_to_doors = defaultdict(int)
for aj, room_idxs in apt_to_rooms.items():
    apt_type = all_aperture_types[aj]
    for ri in room_idxs:
        if apt_type == "window":
            room_to_windows[ri] += 1
        else:
            room_to_doors[ri] += 1

print("Windows per room (including shared assignments):")
for ri in sorted(room_to_windows):
    print(f"  Room {ri}: {room_to_windows[ri]} window connection(s)")
print("Doors per room (including shared assignments):")
for ri in sorted(room_to_doors):
    print(f"  Room {ri}: {room_to_doors[ri]} door connection(s)")
shared_count = sum(1 for rooms in apt_to_rooms.values() if len(rooms) > 1)
print(f"Shared apertures mapped to multiple rooms: {shared_count}")
print(f"Edges built: {len(edges_ra)}")

Windows per room (including shared assignments):
  Room 0: 1 window connection(s)
  Room 1: 1 window connection(s)
  Room 2: 1 window connection(s)
  Room 3: 5 window connection(s)
  Room 5: 2 window connection(s)
  Room 7: 3 window connection(s)
  Room 8: 1 window connection(s)
Doors per room (including shared assignments):
  Room 0: 2 door connection(s)
  Room 1: 2 door connection(s)
  Room 2: 3 door connection(s)
  Room 3: 2 door connection(s)
  Room 4: 1 door connection(s)
  Room 5: 2 door connection(s)
  Room 6: 1 door connection(s)
  Room 7: 4 door connection(s)
  Room 8: 1 door connection(s)
  Room 9: 1 door connection(s)
Shared apertures mapped to multiple rooms: 10
Edges built: 33


In [69]:
bipartite = room_verts + aperture_verts + edges_ra

fig = Topology.Show(
    [cc, all_aperture_cluster] + bipartite,
    faceColor="#4CC9F0", faceOpacity=0.12,
    edgeColor="white", edgeWidth=0.5,
    vertexColor="black",
    vertexSizeKey="size", vertexColorKey="color",
    edgeWidthKey="width", edgeColorKey="color",
    showVertices=True,
    backgroundColor="black",
    width=900, height=700,
    renderer=renderer
)
if fig:
    fig.write_html("Images/09_bipartite_graph.html")